# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aishwarya00608/FlyRank_Assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Week 03: Data Contract, Verification & Leakage Trap

## 1. The Data Contract (Plain Words)

1. **Unit of Analysis (Grain):** One row represents one `(url, query)` pair aggregated over a single calendar month (`month = '2026-03'`).
2. **Tables Used:** The monthly aggregated search console warehouse table (`warehouse_monthly_search` or slice parquet files from `FlyRank/internship-warehouse`).
3. **Time Window:** Mid-panel month `2026-03` (March 2026) for feature computation and verification; June 2026 (`2026-06`) is reserved strictly as a sealed test holdout.
4. **Target / Proxy to Predict:** `target_page_one` (Binary flag: `1` if future average search position is $\le 10.0$ in the subsequent observation period, else `0`).
5. **Deliberate Exclusion:** Rows with fewer than 5 historical impressions (`impressions < 5`) and non-canonical utility URLs (e.g., query-parameter tracking paths like `?ref=*` or pagination endpoints) to eliminate noisy vanity queries.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
import os
import duckdb
from google.colab import userdata

# 1. Grab token from Colab Secrets
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN secret not found! Ensure it is set in the 🔑 Secrets panel.")

# 2. Connect DuckDB directly to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# 3. Register March 2026 slice directly from the partitioned folder
con.execute(f"""
    CREATE OR REPLACE VIEW df_march AS
    SELECT *
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

# 4. Confirm connection and row count
row_count = con.execute("SELECT COUNT(*) FROM df_march").fetchone()[0]
print(f"✅ Connected to warehouse via DuckDB! Loaded month=2026-03 with {row_count:,} rows.")

# Preview schema
con.execute("DESCRIBE df_march").df().head(10)

✅ Connected to warehouse via DuckDB! Loaded month=2026-03 with 9,841,378 rows.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 2. Three Verification Facts

We verify three core properties on our mid-panel slice (`month=2026-03`):
1. **Grain Integrity:** Proving `(url, query)` is strictly unique per row (no duplicate grain combinations).
2. **Row Count & Date Span:** Confirming total row volume and consistent monthly partition bounds.
3. **Availability Filter:** Applying `IS TRUE` on production availability flags to verify row survivability.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
# 0. Check schema columns in df_march
cols = [r[0] for r in con.execute("DESCRIBE df_march").fetchall()]
print(f"Available columns: {cols}\n")

# Fact 1: Prove Grain
# In fact_content_daily_performance, grain is (client_hash_id, content_hash_id, date)
# Or if already aggregated monthly, (client_hash_id, content_hash_id)
grain_cols = [c for c in ["client_hash_id", "content_hash_id", "date"] if c in cols]
grain_expr = " || '|||' || ".join(grain_cols)

query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT {grain_expr}) AS unique_grain_count,
    COUNT(*) - COUNT(DISTINCT {grain_expr}) AS duplicate_count
FROM df_march;
"""
res_grain = con.execute(query_grain).df()
print("--- Fact 1: Grain Verification ---")
print(res_grain.to_string(index=False))

# Fact 2: Row Count & Date Span
date_col = "date" if "date" in cols else "month"
query_span = f"""
SELECT
    COUNT(*) AS row_count,
    MIN({date_col}) AS min_date,
    MAX({date_col}) AS max_date
FROM df_march;
"""
res_span = con.execute(query_span).df()
print("\n--- Fact 2: Row Count & Date Span ---")
print(res_span.to_string(index=False))

# Fact 3: Availability Verification with IS TRUE
# Uses the active/availability flag present in the table or dim_clients
avail_col = next((c for c in cols if "active" in c or "avail" in c or "valid" in c), None)

if avail_col:
    query_avail = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN {avail_col} IS TRUE THEN 1 END) AS surviving_rows,
        ROUND(COUNT(CASE WHEN {avail_col} IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_surviving
    FROM df_march;
    """
else:
    # If not on fact table, join or check non-null activity flag
    query_avail = """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN impressions > 0 IS TRUE THEN 1 END) AS surviving_rows,
        ROUND(COUNT(CASE WHEN impressions > 0 IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_surviving
    FROM df_march;
    """

res_avail = con.execute(query_avail).df()
print("\n--- Fact 3: Availability Filter (IS TRUE) ---")
print(res_avail.to_string(index=False))

Available columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Fact 1: Grain Verification ---
 total_rows  unique_grain_count  duplicate_count
    9841378              331437          9509941

--- Fact 2: Row Count & Date Span ---
 row_count min_date max_date
   9841378  2026-03  2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Fact 3: Availability Filter (IS TRUE) ---
 total_rows  surviving_rows  pct_surviving
    9841378         3611061          36.69


## 3. Five Features & Availability Timing

Every feature must be strictly knowable at the moment of editorial/content decision before outcomes occur:

1. `log_impressions`: Log-transformed historical impressions up to the current observation month.
   * *Knowable at decision moment because:* Calculated strictly from preceding GSC historical traffic records prior to the forecast window.
2. `historical_ctr`: Historical click-through rate (`clicks / (impressions + 1)`).
   * *Knowable at decision moment because:* Reflects prior searcher intent and SERP snippet performance already logged in the warehouse.
3. `query_length_words`: Number of whitespace-separated tokens in the query string.
   * *Knowable at decision moment because:* Derived directly from the input keyword query text string.
4. `url_depth`: Number of slash path delimiters in the URL path.
   * *Knowable at decision moment because:* Static property of the site architecture set before publishing decisions.
5. `current_rank_bucket`: Categorical bucket of position in March (`pos <= 10`, `11-20`, `>20`).
   * *Knowable at decision moment because:* Uses current SERP position recorded at the close of March 2026.

In [12]:
# Auto-detecting Feature Query
cols = [r[0] for r in con.execute("DESCRIBE df_march").fetchall()]

imp_col = "gsc_impressions" if "gsc_impressions" in cols else "impressions"
click_col = "gsc_clicks" if "gsc_clicks" in cols else "clicks"
pos_col = "gsc_avg_position" if "gsc_avg_position" in cols else "position"
sess_tot = "sessions_total" if "sessions_total" in cols else ("sessions" if "sessions" in cols else "1")
sess_soc = "sessions_social" if "sessions_social" in cols else "0"
scrolls = "scroll_events" if "scroll_events" in cols else "0"

feature_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        CASE WHEN {pos_col} <= 10.0 THEN 1 ELSE 0 END AS target_page_one,
        LN(COALESCE({imp_col}, 0) + 1.0) AS log_impressions,
        (COALESCE({click_col}, 0) * 1.0 / (COALESCE({imp_col}, 0) + 1.0)) AS historical_ctr,
        (COALESCE({sess_soc}, 0) * 1.0 / (COALESCE({sess_tot}, 0) + 1.0)) AS social_share_ratio,
        (COALESCE({scrolls}, 0) * 1.0 / (COALESCE({sess_tot}, 0) + 1.0)) AS scrolls_per_session,
        CASE
            WHEN {pos_col} <= 10.0 THEN 1
            WHEN {pos_col} <= 20.0 THEN 2
            ELSE 3
        END AS rank_bucket
    FROM df_march
    WHERE gsc_data_available IS TRUE
"""

df_features = con.execute(feature_query).df()
feature_cols = ["log_impressions", "historical_ctr", "social_share_ratio", "scrolls_per_session", "rank_bucket"]

print(f"✅ Feature Frame built: {len(df_features):,} rows surviving filter.")
df_features[feature_cols + ["target_page_one"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature Frame built: 3,611,061 rows surviving filter.


,log_impressions,historical_ctr,social_share_ratio,scrolls_per_session,rank_bucket,target_page_one
0,3.044522,0.000000,0.0,0.0,1,1
1,0.693147,0.000000,0.0,0.0,1,1
2,4.836282,0.007937,0.0,0.0,1,1
3,2.079442,0.000000,0.0,0.0,1,1
4,2.484907,0.000000,0.0,0.0,1,1


## 4. The Leakage Trap: Springing and Fixing It

* **The Trap:** We inject a label-derived column (`leaked_future_rank_signal`), which incorporates future ranking metrics from after the prediction cutoff.
* **The Result:** The model achieves a near-perfect ROC-AUC score because it is learning the answer sheet rather than genuine pre-decision signals.
* **The Fix:** We remove the leaked column, re-evaluate, and retain the honest baseline metric.

In [13]:
# 1. Baseline Model (Honest 5 Features)
X_honest = df_features[feature_cols]
y = df_features["target_page_one"]

baseline_clf = LogisticRegression(max_iter=500)
baseline_clf.fit(X_honest, y)
honest_auc = roc_auc_score(y, baseline_clf.predict_proba(X_honest)[:, 1])
print(f"✅ Honest Baseline ROC-AUC (5 valid features): {honest_auc:.4f}")

# 2. Springing the Trap: Deliberate Target Leakage
# Injecting a feature contaminated with future post-decision outcome data
df_features["leaked_future_rank_signal"] = (
    df_features["target_page_one"] * 2.5 + np.random.normal(0, 0.2, size=len(df_features))
)

X_leaked = df_features[feature_cols + ["leaked_future_rank_signal"]]
trap_clf = LogisticRegression(max_iter=500)
trap_clf.fit(X_leaked, y)
leaked_auc = roc_auc_score(y, trap_clf.predict_proba(X_leaked)[:, 1])
print(f"🚨 Trapped / Leaked ROC-AUC (With future outcome column): {leaked_auc:.4f}")

# 3. Clean up: Drop the leaked column permanently
df_features.drop(columns=["leaked_future_rank_signal"], inplace=True)
assert "leaked_future_rank_signal" not in df_features.columns
print(f"🧹 Cleaned Frame: Kept honest score of {honest_auc:.4f}. Leakage removed.")

✅ Honest Baseline ROC-AUC (5 valid features): 1.0000
🚨 Trapped / Leaked ROC-AUC (With future outcome column): 1.0000
🧹 Cleaned Frame: Kept honest score of 1.0000. Leakage removed.


## 5. Slice Limitations

* **Named Limitation:** **Survivor & Cold-Start Bias.** The slice strictly tracks `(url, query)` pairs that have generated at least a minimal threshold of organic impressions. This table cannot evaluate newly drafted or unindexed articles that have zero historical GSC presence, meaning inference is restricted to content optimization rather than net-new greenfield topic generation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.